# Epsilon Fund — Combined CPCV Portfolio

Combines **Momentum** and **BB Breakout** CPCV results into a single portfolio-level
return distribution via random path sampling.  Each (strategy, coin) pair is one sleeve;
two-level weighting mirrors the walk-forward portfolio master notebook.

Run **after** completing the per-strategy portfolio CPCV notebooks so each
`*_cpcv.pkl` file exists in the respective `oos/` folders.

---
### What this notebook gives you

| Output | Meaning |
|---|---|
| **Sampled portfolio paths** | N random combinations of per-sleeve paths — each covers the full dataset once |
| **Portfolio distribution** | Sharpe / Calmar / MaxDD distribution across all sampled combinations |
| **Overlap-adjusted CI** | N_eff-corrected confidence intervals (paths share CPCV splits → not independent) |
| **Correlation structure** | How returns co-move across sleeves within each sampled path |
| **Drawdown decomposition** | Which sleeves drive portfolio tail risk |
| **Trade statistics** | Win rate, profit factor, holding period across paths |
| **Optimal weights** | Dirichlet weight search maximising conservative Sharpe floor |

---
### Frequency alignment

Momentum CPCV runs on **daily** bars; BB CPCV runs on **1h** bars.  Both share the
same CPCV structure (N=8, 28 splits, 105 paths), so their path distributions can
be combined.  Before sampling, all path equity curves are normalised to a common
**daily** frequency (last bar of each calendar day) so the return inner-join is
well-defined.  Closed-trade equity uses the native-frequency OOS DataFrames
unchanged.

---
## Imports

In [ ]:
import sys
import os
import io
import contextlib
import pandas as pd
import numpy as np

# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows
# ─────────────────────────────────────────────────────────────────────────────

STRAT_ROOT = os.path.join(ROOT, 'topics', 'momentum', 'strategies')

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))

from cpcv_portfolio import (
    load_asset_cpcv,
    sample_portfolio_paths,
    portfolio_confidence_intervals,
    portfolio_summary,
    asset_correlation_structure,
    worst_drawdown_decomposition,
    optimal_weights,
    extract_portfolio_trade_stats,
    trade_stats_confidence_intervals,
    trade_stats_summary,
    build_representative_oos_dfs,
    coin_inclusion_analysis,
    per_asset_yearly_breakdown,
)
from cpcv_portfolio_visualizer import (
    plot_portfolio_equity_curves,
    plot_portfolio_distribution,
    plot_portfolio_vs_assets,
    plot_portfolio_full_results,
    plot_correlation_structure,
    plot_drawdown_decomposition,
    plot_weight_optimisation,
    plot_yearly_performance,
    plot_trade_statistics,
    plot_asset_yearly_heatmap,
)
from wf_visualizer import plot_closed_trade_equity

---
## Strategy Registry

Declare every CPCV strategy folder you want available.  Each entry points to the
`oos/` directory produced by that strategy's per-asset CPCV notebooks and defines
how pkl files are named for that strategy.

| Tag | Strategy | Bar frequency | pkl naming |
|---|---|---|---|
| `momentum` | Momentum (wf_testing_2) | Daily | `{coin.lower()}usdt_cpcv.pkl` |
| `bb` | BB Breakout | 1 h | `{coin.lower()}usdt_1h_cpcv.pkl` |

In [ ]:
# ── strategy registry ─────────────────────────────────────────────────────────
# Each strategy entry: folder path + pkl naming convention for its assets.
STRATEGIES = {
    'momentum': {
        'folder':   os.path.join(STRAT_ROOT, 'momentum_cpcv', 'oos'),
        'pkl_name': lambda coin: f'{coin.lower()}usdt_cpcv.pkl',
    },
    'bb': {
        'folder':   os.path.join(STRAT_ROOT, 'bb_cpcv', 'oos'),
        'pkl_name': lambda coin: f'{coin.lower()}usdt_1h_cpcv.pkl',
    },
    # ── add more strategies here ──────────────────────────────────────────────
    # 'new_strat': {
    #     'folder':   os.path.join(STRAT_ROOT, 'new_strat_cpcv', 'oos'),
    #     'pkl_name': lambda coin: f'{coin.lower()}usdt_cpcv.pkl',
    # },
}

# ── discover available sleeves from existing pkl files ───────────────────────
print('Available CPCV pkls per strategy:')
for tag, cfg in STRATEGIES.items():
    folder = cfg['folder']
    if not os.path.isdir(folder):
        print(f'  {tag}: folder not found — {folder}')
        continue
    found = sorted(f for f in os.listdir(folder) if f.endswith('_cpcv.pkl'))
    print(f'  {tag} ({os.path.basename(os.path.dirname(folder))}): {found}')

---
## Sleeve Selection

Pick which **(strategy, coin)** pairs to include.  Each entry becomes one **sleeve**
in the combined portfolio.  Labels appear in all plots and tables.

**Two-level weighting** mirrors the portfolio master:
1. `STRATEGY_WEIGHTS` — how much of the book each strategy gets.
2. `COIN_WEIGHTS` — within each strategy, how the allocation splits across coins.
   `None` for a strategy = equal-weight across its selected sleeves.

Final sleeve weight = `STRATEGY_WEIGHTS[strat] × coin_weight_within_strat`.
Both dicts are auto-normalised, so values only need to be proportional.

```python
# Example — 60/40 strategy split, momentum overweighted to BTC:
STRATEGY_WEIGHTS = {'momentum': 0.60, 'bb': 0.40}
COIN_WEIGHTS = {
    'momentum': {'BTC': 0.30, 'ETH': 0.20, 'SOL': 0.20, 'XRP': 0.15, 'ADA': 0.10, 'AVAX': 0.05},
    'bb':       None,   # equal across bb sleeves
}
```

In [ ]:
# ── sleeve selection ─────────────────────────────────────────────────────────
# label → (strategy_tag, coin)
SELECTION = {
    # ── Momentum (daily) ─────────────────────────────────────────────────────
    'BTC_m':   ('momentum', 'BTC'),
    'ETH_m':   ('momentum', 'ETH'),
    'SOL_m':   ('momentum', 'SOL'),
    'XRP_m':   ('momentum', 'XRP'),
    'ADA_m':   ('momentum', 'ADA'),
    'AVAX_m':  ('momentum', 'AVAX'),
    # ── BB Breakout (1h) ─────────────────────────────────────────────────────
    'BTC_bb':  ('bb', 'BTC'),
    'ETH_bb':  ('bb', 'ETH'),
#    'ADA_bb':  ('bb', 'ADA'),
    'AVAX_bb': ('bb', 'AVAX'),
#    'LINK_bb': ('bb', 'LINK'),
    'POL_bb': ('bb', 'POL'),

}

# ── which sleeves to include in the portfolio (None = all) ───────────────────
SHOW_SLEEVES = list(SELECTION.keys())   # or a manual subset

# ── CPCV sampling ────────────────────────────────────────────────────────────
N_SAMPLES = 2000
SEED      = 42

# ── two-level weighting ──────────────────────────────────────────────────────
# 1) across strategies — None = equal-weight across strategies in SHOW_SLEEVES
STRATEGY_WEIGHTS = {'momentum': 0.4, 'bb': 0.6}

# 2) within each strategy — None (or strat missing) = equal-weight across that
#    strategy's sleeves in SHOW_SLEEVES
COIN_WEIGHTS = None
# e.g. COIN_WEIGHTS = {
#     'momentum': {'BTC': 0.25, 'ETH': 0.20, 'SOL': 0.20, 'XRP': 0.15, 'ADA': 0.10, 'AVAX': 0.10},
#     'bb':       {'BTC': 0.35, 'ETH': 0.30, 'ADA': 0.20, 'AVAX': 0.15},
# }

# WF Sharpe reference lines in the distribution chart (optional)
WF_SHARPES = {
    # 'BTC_m': 1.45,   # ← fill in from walk-forward notebook
}

# ── compute flat sleeve weights from the two levels ──────────────────────────
def _norm(d):
    s = sum(d.values())
    return {k: v / s for k, v in d.items()} if s > 0 else d

_strats_in_view = {SELECTION[s][0] for s in SHOW_SLEEVES}
_sw = _norm(STRATEGY_WEIGHTS or {t: 1.0 for t in _strats_in_view})

WEIGHTS = {}
for tag in _strats_in_view:
    sleeves_of_tag = [s for s in SHOW_SLEEVES if SELECTION[s][0] == tag]
    cw = (COIN_WEIGHTS or {}).get(tag)
    if cw is None:
        cw_norm = {SELECTION[s][1]: 1.0 / len(sleeves_of_tag) for s in sleeves_of_tag}
    else:
        cw_norm = _norm({SELECTION[s][1]: cw.get(SELECTION[s][1], 0) for s in sleeves_of_tag})
    for s in sleeves_of_tag:
        WEIGHTS[s] = _sw[tag] * cw_norm[SELECTION[s][1]]

print('Sleeve weights (strategy × coin):')
for s in SHOW_SLEEVES:
    tag, coin = SELECTION[s]
    print(f'  {s:<10}  strat={tag:<10} coin={coin:<5}  w={WEIGHTS[s]*100:>6.2f}%')
print(f'  {""}  total = {sum(WEIGHTS.values())*100:.2f}%')

---
## Load CPCV Pkls

Each strategy's pkls are loaded and validated independently (N / n_splits / n_paths
must match within a strategy).  The two strategy dicts are then merged.

Because Momentum runs on **daily** bars and BB Breakout on **1 h** bars, all path
equity curves are resampled to a common **daily** frequency before portfolio path
sampling.  This is lossless in expectation: the daily return equals the compounded
product of all intraday bar returns for that day.

> `split_results` (raw OOS DataFrames) are **not resampled** — they stay at native
> frequency for the closed-trade equity chart.

In [ ]:
# ── build pkl path dicts per strategy ────────────────────────────────────────
_pkl_by_strategy = {tag: {} for tag in STRATEGIES}
for label in SHOW_SLEEVES:
    tag, coin = SELECTION[label]
    if tag not in STRATEGIES:
        print(f'  [skip] {label}: unknown strategy tag "{tag}"')
        continue
    cfg  = STRATEGIES[tag]
    path = os.path.join(cfg['folder'], cfg['pkl_name'](coin))
    if not os.path.exists(path):
        print(f'  [missing] {label}: {path}')
    else:
        _pkl_by_strategy[tag][label] = path

# ── load + validate per strategy ─────────────────────────────────────────────
asset_results_raw = {}
for tag, pkl_dict in _pkl_by_strategy.items():
    if not pkl_dict:
        print(f'\n[{tag}] no pkls found — skipping')
        continue
    print(f'\n── {tag} ──')
    strat_results = load_asset_cpcv(pkl_dict)
    asset_results_raw.update(strat_results)

print(f'\nTotal sleeves loaded: {len(asset_results_raw)}')

# ── remove any sleeves that failed to load ────────────────────────────────────
WEIGHTS = {k: v for k, v in WEIGHTS.items() if k in asset_results_raw}
# renormalise after dropping missing
WEIGHTS = _norm(WEIGHTS)

# ── frequency normalisation helper ───────────────────────────────────────────
def _normalise_equity_to_daily(asset_results: dict) -> dict:
    """
    Return a copy of asset_results with all path equity_curves normalised to
    tz-naive daily frequency AND per-path metrics (sharpe, calmar, max_dd,
    total_return) recomputed at daily frequency.

    Intraday assets (gap < 20 h) are downsampled via last-bar-of-day.
    Daily assets are only tz-stripped / normalised if needed.

    Why recompute metrics?
    ─────────────────────
    After normalising BB equity curves from 1 h to daily, the stored p['sharpe']
    values are stale (they were computed at 1 h frequency on the raw paths).
    portfolio_summary() reads p['sharpe'] directly, while sample_portfolio_paths()
    uses the equity curves — so without recomputation, the "Median Sharpe" column
    in the weights table shows native-1H values (~1.56 for BTC_bb) while the
    portfolio distribution shows the daily-computed value.  Recomputing makes
    every Sharpe figure in this notebook consistent at the same daily basis.

    Shares split_results / split_assignments with the input dict (shallow copy)
    so memory stays low and build_representative_oos_dfs still has access to
    the raw OOS DataFrames.
    """
    PPY_DAILY = 365   # guaranteed after daily resample

    def _metrics_from_daily_eq(ec):
        """Recompute (sharpe, calmar, max_dd, total_return) from a daily equity curve."""
        r   = ec.pct_change().fillna(0)
        n   = len(r)
        if n < 2:
            return 0.0, 0.0, 0.0, 0.0
        tot     = float((1 + r).prod() - 1)
        std     = float(r.std())
        sh      = float(r.mean() / std * np.sqrt(PPY_DAILY)) if std > 0 else 0.0
        eq_c    = (1 + r).cumprod()
        run_max = eq_c.cummax()
        dd      = float(((eq_c - run_max) / run_max).min())
        n_years = n / PPY_DAILY
        ann_ret = (1 + tot) ** (1.0 / n_years) - 1 if n_years > 0 else 0.0
        cal     = float(ann_ret / abs(dd)) if (n_years > 0 and dd != 0) else 0.0
        return sh, cal, dd, tot

    out = {}
    for asset, results in asset_results.items():
        new_paths = []
        for path in results['paths']:
            ec = path.get('equity_curve')
            if ec is None or len(ec) < 2:
                new_paths.append(path)
                continue

            new_path = dict(path)   # shallow copy — split_assignments etc. shared

            med_gap_h = (ec.index.to_series().diff().dropna()
                         .median().total_seconds() / 3600)

            if med_gap_h < 20:   # intraday (hourly, 4 h, …)
                ec = ec.resample('D').last().dropna()

            if getattr(ec.index, 'tz', None) is not None:
                ec = ec.copy()
                ec.index = ec.index.tz_localize(None)

            ec.index = ec.index.normalize()
            new_path['equity_curve'] = ec

            # ── recompute metrics at daily frequency ──────────────────────────
            sh, cal, dd, tot = _metrics_from_daily_eq(ec)
            new_path['sharpe']       = sh
            new_path['calmar']       = cal
            new_path['max_dd']       = dd
            new_path['total_return'] = tot

            new_paths.append(new_path)

        new_r = dict(results)    # shallow copy — split_results still shared
        new_r['paths'] = new_paths
        out[asset] = new_r

    return out

# ── apply normalisation ───────────────────────────────────────────────────────
asset_results = _normalise_equity_to_daily(asset_results_raw)
print('\nEquity curves normalised to daily; per-path metrics recomputed at daily freq (ppy=365).')
print('Note: BB sleeves show lower Sharpe here than in the BB-only CPCV notebook.')
print('      That notebook uses native 1h bars (ppy=8760) — the difference is expected.')


---
## Individual Sleeve Results (reference)

One-line summary per sleeve before combining — confirm each asset's CPCV is healthy
before interpreting portfolio-level results.

In [ ]:
print(f'  {"Sleeve":<10} {"Strat":<10} {"Coin":<6} {"Paths":>7} {"Median Sharpe":>14} {"Weight":>8}')
print(f'  {"-"*10} {"-"*10} {"-"*6} {"-"*7} {"-"*14} {"-"*8}')
for label in SHOW_SLEEVES:
    if label not in asset_results:
        continue
    tag, coin = SELECTION[label]
    results = asset_results[label]
    path_sharpes = [p['sharpe'] for p in results['paths'] if p.get('sharpe') is not None]
    median_sh = float(np.median(path_sharpes)) if path_sharpes else float('nan')
    print(f'  {label:<10} {tag:<10} {coin:<6} {len(path_sharpes):>7} {median_sh:>14.2f} {WEIGHTS.get(label, 0)*100:>7.2f}%')

---
## Per-Strategy Isolation

Run each strategy in isolation (equal coin weights within that strategy) and compare
with the combined portfolio.  Directly answers: **does combining Momentum and BB add
value over running either strategy alone?**

| Metric | Momentum-only | BB-only | Combined |
|---|---|---|---|
| Mean Sharpe | ... | ... | ... |
| Conservative floor | ... | ... | ... |
| Mean Max DD | ... | ... | ... |

- A **higher combined floor** than both individual floors confirms genuine CPCV-validated
  diversification benefit — not just averaging two signals.
- If one strategy's floor is already above the combined, the weaker strategy is dragging
  it down and should be re-examined via `STRATEGY_WEIGHTS`.
- `n_samples=500` is sufficient for ranking; the full 2000-path run happens in the main
  portfolio sampling cell.

> **All figures in this notebook use daily-normalised equity curves (ppy=365).**
> Equity curves are resampled to daily (last bar of each calendar day) and all
> per-path metrics (Sharpe, Calmar, Max DD) are recomputed at daily frequency.
> Isolation Sharpe values are therefore directly comparable to the sweep table,
> the combined column, and the per-sleeve vlines in the Sharpe distribution chart.
>
> **Why BB isolation Sharpe is lower than the BB-only CPCV notebook:**  
> The BB-only notebook (`bb_cpcv/portfolio_cpcv.ipynb`) uses native **1 h** bars
> (ppy = 8760) and shows Mean Sharpe ≈ 2.69.  This notebook normalises BB to daily
> (ppy = 365); the Sharpe decreases because the daily-compounded equity curve has
> fewer periods per year.  The two numbers measure the same underlying strategy at
> different time resolutions — neither is wrong.  The difference is expected and
> does not indicate a calculation error.

In [ ]:
_N_ISO = 500   # path count for isolation runs (fast; enough for ranking)

_strat_tags   = list({SELECTION[s][0] for s in WEIGHTS})
_sleeves_by_s = {tag: [s for s in WEIGHTS if SELECTION[s][0] == tag]
                 for tag in _strat_tags}

def _iso_weights(sleeves):
    n = len(sleeves)
    return {s: 1.0 / n for s in sleeves}

def _iso_stats(sleeves):
    """
    Run an isolation back-test for a set of sleeves using daily-normalised equity
    curves (asset_results, ppy=365).  This keeps isolation Sharpe values on the same
    basis as the sweep table, the combined portfolio, and the per-sleeve vlines in
    the Sharpe distribution chart — so all numbers in this notebook are directly
    comparable regardless of each strategy's native bar frequency.
    """
    w = _iso_weights(sleeves)
    w = {k: v for k, v in w.items() if k in asset_results}
    if not w:
        return None, None
    w = _norm(w)
    with contextlib.redirect_stdout(io.StringIO()):
        paths = sample_portfolio_paths(asset_results, w, n_samples=_N_ISO, seed=SEED)
        ci_   = portfolio_confidence_intervals(paths, asset_results)
    sharpes = [p['sharpe'] for p in paths]
    mdd     = [p['max_dd'] for p in paths]
    calmar  = [p['calmar'] for p in paths]
    return {
        'mean_sharpe':  float(np.mean(sharpes)),
        'med_sharpe':   float(np.median(sharpes)),
        'floor':        ci_['sharpe']['conservative_lower_bound'],
        'mean_dd':      float(np.mean(mdd)),
        'mean_calmar':  float(np.mean(calmar)),
        'n_sleeves':    len(w),
    }, paths

# ── run isolations ────────────────────────────────────────────────────────────
isolation_results = {}
for tag, sleeves in _sleeves_by_s.items():
    print(f'Running {tag}-only isolation ({len(sleeves)} sleeves, n={_N_ISO} paths)…')
    stats, _ = _iso_stats(sleeves)
    isolation_results[tag] = stats

print()

# ── print partial table (combined row added after Portfolio Summary runs) ─────
W = 72
print('═' * W)
print(f'  STRATEGY ISOLATION COMPARISON  ({_N_ISO} paths each, daily freq, equal weights within strategy)')
print('─' * W)

tags_ordered = sorted(isolation_results.keys())
header = f'  {"Metric":<26}' + ''.join(f'  {t.capitalize():<14}' for t in tags_ordered) + '  {"Combined":<12}'
print(header)
print('─' * W)

_isolation_combined_stats = None   # populated in the cell after Portfolio Summary

_metrics = [
    ('Mean Sharpe',        'mean_sharpe',  False),
    ('Median Sharpe',      'med_sharpe',   False),
    ('Conservative Floor', 'floor',        False),
    ('Mean Max DD',        'mean_dd',      True ),
    ('Mean Calmar',        'mean_calmar',  False),
]

for label, key, pct in _metrics:
    row = f'  {label:<26}'
    for tag in tags_ordered:
        v = isolation_results[tag][key] if isolation_results.get(tag) else float('nan')
        row += f'  {v*100:>13.1f}%' if pct else f'  {v:>14.2f}'
    row += '  (see summary)'
    print(row)

print('═' * W)
print('  ↑ Combined row populated after Portfolio Summary cell runs.')

---
## Portfolio Path Sampling

For each of `N_SAMPLES` draws, one valid CPCV path is independently sampled per
sleeve.  Returns are inner-joined on the common **daily** date index and combined
using `WEIGHTS`.  Metrics are computed on the weighted portfolio daily return series.

| Parameter | Description |
|---|---|
| `N_SAMPLES` | Random combinations to draw — more samples reduce sampling noise |
| `SEED` | RNG seed — fix at 42 for reproducibility |

In [ ]:
portfolio_paths = sample_portfolio_paths(
    asset_results, WEIGHTS,
    n_samples=N_SAMPLES, seed=SEED,
)
print(f'Sampled {len(portfolio_paths)} portfolio paths')

---
## Portfolio Summary

Three tables:

1. **Path distribution** — Mean / Std / Min / Q25 / Median / Q75 / Max / IQR across
   Sharpe, Calmar, Max DD, and Return for all sampled portfolio paths.
2. **Confidence intervals** — naive (anticonservative) and overlap-adjusted (conservative).
   The **conservative lower bound** is the most cautious estimate of mean Sharpe.
3. **Sleeve weights** — per-sleeve weight, median Sharpe (daily-normalised), and
   normalised contribution.

> **Note on N_eff:** overlap is estimated as the average weighted per-asset overlap
> fraction across all path pairs — the same formula used in per-asset CPCV notebooks.
> This gives N_eff ≈ 5–8 (consistent with individual strategy notebooks).  The
> adjusted CI is meaningfully wide but not pathological; use the conservative floor
> as your hurdle-rate benchmark.

In [ ]:
ci = portfolio_confidence_intervals(portfolio_paths, asset_results)
portfolio_summary(portfolio_paths, ci, WEIGHTS, asset_results=asset_results)

In [ ]:
# ── complete isolation comparison (now with combined stats) ───────────────────
_combined_sharpes = [p['sharpe'] for p in portfolio_paths]
_combined_mdd     = [p['max_dd'] for p in portfolio_paths]
_combined_calmar  = [p['calmar'] for p in portfolio_paths]
_isolation_combined_stats = {
    'mean_sharpe':  float(np.mean(_combined_sharpes)),
    'med_sharpe':   float(np.median(_combined_sharpes)),
    'floor':        ci['sharpe']['conservative_lower_bound'],
    'mean_dd':      float(np.mean(_combined_mdd)),
    'mean_calmar':  float(np.mean(_combined_calmar)),
}

W = 72
print('═' * W)
print(f'  STRATEGY ISOLATION vs COMBINED  (isolation={_N_ISO} paths, combined={N_SAMPLES} paths)')
print('─' * W)
print(f'  {"Metric":<26}' + ''.join(f'  {t.capitalize():<14}' for t in tags_ordered) + f'  {"Combined":<12}')
print('─' * W)

for label, key, pct in _metrics:
    row = f'  {label:<26}'
    for tag in tags_ordered:
        v = isolation_results[tag][key] if isolation_results.get(tag) else float('nan')
        row += f'  {v*100:>13.1f}%' if pct else f'  {v:>14.2f}'
    v_comb = _isolation_combined_stats[key]
    row += f'  {v_comb*100:>11.1f}%' if pct else f'  {v_comb:>12.2f}'
    print(row)

print('═' * W)

_best_iso_floor = max(v['floor'] for v in isolation_results.values() if v)
_comb_floor     = _isolation_combined_stats['floor']
_delta          = _comb_floor - _best_iso_floor
marker = 'POSITIVE' if _delta >= 0 else 'NEGATIVE'
print(f'\n  Diversification signal: combined floor − best-strategy floor = {_delta:+.3f}  [{marker}]')
if _delta >= 0:
    print('  → Combining raises (or maintains) the conservative Sharpe floor. Diversification is real.')
else:
    print('  → Combined floor is below the best individual strategy. Consider increasing that strategy weight.')

---
## Strategy Split Sweep

Sweeps the **Momentum % vs BB %** allocation from 0 → 100 in 5 % steps, holding
coin weights equal within each strategy at each step.  Produces CPCV path metrics at
every split so you can see the optimal strategy ratio without running the full
`optimal_weights` search.

This is the CPCV equivalent of portfolio_master's `sweep_momentum_strategy`.

| Column | Meaning |
|---|---|
| `mom%` / `bb%` | Strategy allocation at this step |
| `Mean Sh` | Mean Sharpe across all sampled paths |
| `Med Sh` | Median Sharpe |
| `Floor` | Overlap-adjusted conservative Sharpe lower bound (key metric) |
| `Mean DD` | Mean maximum drawdown across paths |
| `Mean Calmar` | Mean Calmar ratio |

> `n=500` paths per step is enough for reliable ranking.  The ← current row shows
> where your configured `STRATEGY_WEIGHTS` sits on this curve.  All rows use
> daily-normalised equity curves (ppy=365), so they are directly comparable to the
> isolation table and the combined portfolio summary above.

In [ ]:
_N_SWEEP  = 500    # paths per sweep step — fast enough, good enough for ranking
_STEP     = 5      # % increment

_mom_sleeves = [s for s in WEIGHTS if SELECTION[s][0] == 'momentum']
_bb_sleeves  = [s for s in WEIGHTS if SELECTION[s][0] == 'bb']
_n_mom_sl    = len(_mom_sleeves)
_n_bb_sl     = len(_bb_sleeves)

_cur_mom_pct = round(_sw.get('momentum', 0) * 100)

sweep_rows = []
for mom_pct in range(0, 101, _STEP):
    bb_pct = 100 - mom_pct
    w = {}
    if mom_pct > 0 and _n_mom_sl > 0:
        for s in _mom_sleeves:
            w[s] = (mom_pct / 100) / _n_mom_sl
    if bb_pct > 0 and _n_bb_sl > 0:
        for s in _bb_sleeves:
            w[s] = (bb_pct / 100) / _n_bb_sl
    w = {k: v for k, v in w.items() if k in asset_results}
    if not w:
        continue

    with contextlib.redirect_stdout(io.StringIO()):
        _p  = sample_portfolio_paths(asset_results, w, n_samples=_N_SWEEP, seed=SEED)
        _ci = portfolio_confidence_intervals(_p, asset_results)

    _sh  = [p['sharpe'] for p in _p]
    _dd  = [p['max_dd'] for p in _p]
    _cal = [p['calmar'] for p in _p]
    sweep_rows.append({
        'mom_pct':     mom_pct,
        'bb_pct':      bb_pct,
        'mean_sh':     float(np.mean(_sh)),
        'med_sh':      float(np.median(_sh)),
        'floor':       _ci['sharpe']['conservative_lower_bound'],
        'mean_dd':     float(np.mean(_dd)),
        'mean_calmar': float(np.mean(_cal)),
        'is_current':  abs(mom_pct - _cur_mom_pct) <= _STEP / 2,
    })

_best_floor  = max(sweep_rows, key=lambda r: r['floor'])
_best_sharpe = max(sweep_rows, key=lambda r: r['mean_sh'])

W = 80
print('═' * W)
print(f'  STRATEGY SPLIT SWEEP  (equal coin weights within each strategy, n={_N_SWEEP} paths/step)')
print('─' * W)
print(f'  {"mom%":>5}  {"bb%":>4}  {"Mean Sh":>8}  {"Med Sh":>7}  {"Floor":>7}  {"Mean DD":>8}  {"Mean Cal":>9}')
print(f'  {"-"*5}  {"-"*4}  {"-"*8}  {"-"*7}  {"-"*7}  {"-"*8}  {"-"*9}')

for r in sweep_rows:
    tags = []
    if r is _best_floor:  tags.append('best floor')
    if r is _best_sharpe and r is not _best_floor: tags.append('best Sharpe')
    if r['is_current']:   tags.append('current')
    suffix = f'  ← {", ".join(tags)}' if tags else ''
    print(f'  {r["mom_pct"]:>5}  {r["bb_pct"]:>4}  {r["mean_sh"]:>8.2f}  '
          f'{r["med_sh"]:>7.2f}  {r["floor"]:>7.2f}  '
          f'{r["mean_dd"]*100:>7.1f}%  {r["mean_calmar"]:>9.2f}{suffix}')

print('═' * W)
print(f'\n  Best floor  : {_best_floor["mom_pct"]}% momentum / {_best_floor["bb_pct"]}% bb'
      f'  →  floor={_best_floor["floor"]:.3f}')
print(f'  Best Sharpe : {_best_sharpe["mom_pct"]}% momentum / {_best_sharpe["bb_pct"]}% bb'
      f'  →  mean Sharpe={_best_sharpe["mean_sh"]:.3f}')
print(f'  Current     : {_cur_mom_pct}% momentum / {100-_cur_mom_pct}% bb')
print()
print('  → Use these results to refine STRATEGY_WEIGHTS in the config cell above.')

---
## Portfolio Path Distribution

Three charts:

- **Equity curves** — up to 200 sampled portfolio paths (blue, semi-transparent),
  mean path (amber), and min–max envelope.  All paths are base-100.
- **Sharpe histogram** — distribution of sampled portfolio Sharpe ratios.  The blue
  shaded band is the overlap-adjusted 95% CI; the dashed red line is the conservative
  floor.  Dotted coloured vlines show each sleeve's individual CPCV median Sharpe.
- **Sleeve vs portfolio box plots** — per-sleeve CPCV path Sharpe distributions (blue)
  sorted by median, plus the sampled portfolio distribution (amber).

In [ ]:
_wf = {k: v for k, v in WF_SHARPES.items() if v is not None}

plot_portfolio_full_results(
    portfolio_paths,
    asset_results,
    ci_results  = ci,
    weights     = WEIGHTS,
    wf_sharpes  = _wf if _wf else None,
);

---
## Closed-Trade Equity

Same underlying strategy and **the same closed-trade-only sizing model** as the path
distribution above — position notional is scaled by realized (closed) equity only;
unrealized gains never inflate future trade sizes in either chart.

The only difference is **temporal resolution**:

| Chart | Curve moves | Paths |
|---|---|---|
| CPCV path distribution (above) | Every bar — intra-trade fluctuations visible | 2000 random path combinations |
| Closed-trade equity (below) | Only on trade exits — flat between trades | One representative path per sleeve |

Each sleeve uses its **representative path**: the CPCV path whose Sharpe is closest
to that sleeve's median across all 105 paths.  The weighted average of per-sleeve
exit-by-exit returns produces the portfolio closed-trade curve.

> BB sleeves use native **1 h** OOS DataFrames; Momentum sleeves use native **daily**
> OOS DataFrames.  Both are handled correctly by `plot_closed_trade_equity`.

> The closed-trade view isolates trade-by-trade realized P&L with no intra-trade noise.  
> Read it alongside the path distribution, not instead of it.

In [ ]:
COST_CT = 0.001   # keep aligned with cost used in plot_portfolio_full_results above

# build_representative_oos_dfs uses asset_results_raw so OOS DataFrames
# remain at their native frequency (daily for momentum, 1h for BB).
rep_dfs = build_representative_oos_dfs(asset_results_raw, WEIGHTS)

plot_closed_trade_equity(
    rep_dfs,
    weights = WEIGHTS,
    cost    = COST_CT,
);

---
## Asset Correlation Structure

Pairwise Pearson return correlations across all sampled portfolio paths.
Each path gives one correlation estimate per sleeve pair; the distribution across
paths reveals how stable (or regime-dependent) correlations are.

- **Violin per pair** — full distribution of per-path correlations; pairs coloured
  by median: green < 0.3, amber < 0.5, red ≥ 0.5.  Dashed line at 0.7 flags
  high-correlation paths.
- **Heatmap** — symmetric matrix of mean pairwise correlations (±std in cell text).

Same-coin cross-strategy pairs (e.g. `BTC_m` vs `BTC_bb`) are expected to be
moderately correlated — they trade the same underlying but on different timeframes
and signals, so correlation < 1 is normal and desirable.

In [ ]:
corr_results = asset_correlation_structure(
    portfolio_paths,
    asset_results,
    WEIGHTS,
)

plot_correlation_structure(corr_results);

---
## Sleeve Inclusion Analysis

For each sleeve, computes three CPCV-validated metrics:

| Column | Meaning |
|---|---|
| `ind_floor` | That sleeve's own overlap-adjusted conservative Sharpe floor (run in isolation) |
| `excl_floor` | Portfolio conservative floor when this sleeve is **excluded** (equal-weight of the rest) |
| `marg_contribution` | `full_portfolio_floor − excl_floor` — how much this sleeve adds to the portfolio floor |
| `recommendation` | `include` / `marginal` / `exclude` based on both thresholds |

**Decision rules:**
- `include` — individual floor ≥ threshold **and** marginal contribution ≥ 0
- `exclude` — individual floor < threshold **or** marginal contribution < −0.05
- `marginal` — otherwise

This is the CPCV equivalent of portfolio_master's per-coin isolation table, but
evaluated against the **overlap-adjusted floor** rather than a single-run Sharpe.
Use the `exclude` recommendations to trim dead-weight sleeves before running the
full `optimal_weights` search.

In [ ]:
inclusion_df = coin_inclusion_analysis(
    asset_results,
    base_weights            = WEIGHTS,
    n_samples               = 500,    # fast; full N_SAMPLES in main run
    seed                    = SEED,
    sharpe_floor_threshold  = 0.5,    # min acceptable individual floor
)

---
## Worst Drawdown Decomposition

Identifies the `worst_pct` of portfolio paths by maximum drawdown and decomposes
each drawdown period into per-sleeve contributions.

- **Contribution** = `weight × cumulative return over the drawdown window`
- **Pct of DD** = sleeve contribution as a fraction of total portfolio drawdown
- **Primary driver** = sleeve with the highest mean `|pct_of_dd|` across worst paths

Use this to understand which sleeves (and which strategy) dominate portfolio risk
in tail scenarios.  If momentum sleeves consistently drive worst paths, consider
reducing `STRATEGY_WEIGHTS['momentum']`.

In [ ]:
dd_results = worst_drawdown_decomposition(
    portfolio_paths,
    asset_results,
    WEIGHTS,
    worst_pct=0.10,
)

plot_drawdown_decomposition(dd_results, WEIGHTS);

---
## Trade Statistics

Extracts trade-level data from the per-sleeve OOS strategy DataFrames stored
inside each CPCV split result.  For every sampled portfolio path the following
are aggregated across all sleeves and all CPCV groups assigned to that path:

| Statistic | Description |
|---|---|
| **Win rate** | Fraction of trades with positive PnL |
| **Avg trade return** | Mean PnL across all trades |
| **Profit factor** | Gross profit / gross loss |
| **Avg holding period** | Mean bars per trade |
| **Trades per year** | Annualised trade count |

Yearly stats (return, Sharpe, MaxDD, trade count) are computed from
`portfolio_returns` and the per-group trade DataFrames.

In [ ]:
trade_stats = extract_portfolio_trade_stats(portfolio_paths, asset_results, WEIGHTS)

trade_cis = trade_stats_confidence_intervals(trade_stats, portfolio_paths, asset_results)

trade_stats_summary(trade_stats, trade_cis)

In [ ]:
plot_trade_statistics(trade_stats, trade_cis);
plot_yearly_performance(trade_stats, trade_cis);

---
## Portfolio Yearly Performance (Path Distribution)

Computes annual returns for every calendar year directly from each sampled portfolio
path's `portfolio_returns` series.  Shows the **full cross-path distribution** — not
just the mean — so you can see how much the outcome varies year to year.

| Column | Meaning |
|---|---|
| `Mean` | Mean annual return across all paths that contained that year |
| `Median` | Median annual return |
| `Q25 / Q75` | Inter-quartile range across paths |
| `% Positive` | Fraction of paths with positive annual return that year |
| `Best / Worst` | Extreme path outcomes |

> This complements `plot_yearly_performance` in the trade statistics section above,
> which uses the trade-level yearly breakdown.  This view uses the bar-level portfolio
> returns and captures the full path distribution uncertainty per year.

In [ ]:
yearly = per_asset_yearly_breakdown(portfolio_paths, asset_results, WEIGHTS)

# ── portfolio-level table ─────────────────────────────────────────────────────
port_yr = yearly['portfolio_year_summary'].sort_values('year')
print(f'  {"Year":>5}  {"Mean Ret":>9}  {"Median":>8}  {"Std":>7}  {"% Pos":>7}  {"Sh":>6}')
print(f'  {"-"*5}  {"-"*9}  {"-"*8}  {"-"*7}  {"-"*7}  {"-"*6}')
for _, r in port_yr.iterrows():
    print(f'  {int(r.year):>5}  {r.mean_return*100:>8.1f}%  {r.median_return*100:>7.1f}%  '
          f'{r.std_return*100:>6.1f}%  {r.pct_paths_positive*100:>6.0f}%  {r.mean_sharpe:>6.2f}')

# ── full heatmap (per-sleeve + Portfolio row, with % positive) ────────────────
plot_asset_yearly_heatmap(yearly);

---
## Optimal Weight Search

Samples `N_WEIGHT_CONFIGS` random weight configurations using Dirichlet sampling
and evaluates each via `n_samples` portfolio paths.

| Objective | Meaning |
|---|---|
| `max_conservative_sharpe` | Maximise the overlap-adjusted conservative Sharpe lower bound |
| `max_mean_sharpe` | Maximise mean portfolio Sharpe across all sampled paths |
| `min_max_dd` | Minimise mean maximum drawdown |

The equal-weight allocation across all sleeves is always evaluated as config 0 for
reference.  Weights are per-sleeve — use the output to inform both `STRATEGY_WEIGHTS`
and `COIN_WEIGHTS` in the config above.

> **`min_weight` / `max_weight` guidance for 10 sleeves:** with 10 sleeves the
> natural equal weight is 10%.  Setting `max_weight = 0.30` (3× equal weight)
> keeps allocations balanced; `max_weight = 0.60` would allow one sleeve to take
> 60% of the book, which typically overfits.  Default here is 0.30.
>
> If the strategy split sweep (above) showed a clear dominant strategy, consider
> running this search on that strategy's sleeves only by reducing `SHOW_SLEEVES`
> and re-running from the config cell.

In [ ]:
N_WEIGHT_CONFIGS = 200   # increase to 500 for a thorough search

optimal_results = optimal_weights(
    asset_results,
    n_samples        = 300,
    n_weight_configs = N_WEIGHT_CONFIGS,
    min_weight       = 0.03,   # 3% floor — allows small tilts
    max_weight       = 0.30,   # 30% ceiling — prevents single-sleeve concentration
)

plot_weight_optimisation(optimal_results, list(WEIGHTS.keys()));

---
## Save All Results

Pickle all portfolio-level outputs for downstream use (e.g. comparing multiple
strategy/weight configurations without re-running the full analysis).

In [ ]:
SAVE_DIR = os.path.join(ROOT, 'topics', 'momentum', 'results', 'oos')
os.makedirs(SAVE_DIR, exist_ok=True)

pd.to_pickle(portfolio_paths,        os.path.join(SAVE_DIR, 'combined_cpcv_paths.pkl'))
pd.to_pickle(ci,                     os.path.join(SAVE_DIR, 'combined_cpcv_ci.pkl'))
pd.to_pickle(corr_results,           os.path.join(SAVE_DIR, 'combined_cpcv_corr.pkl'))
pd.to_pickle(dd_results,             os.path.join(SAVE_DIR, 'combined_cpcv_dd.pkl'))
pd.to_pickle(optimal_results,        os.path.join(SAVE_DIR, 'combined_cpcv_weights.pkl'))
pd.to_pickle(trade_stats,            os.path.join(SAVE_DIR, 'combined_cpcv_trade_stats.pkl'))
pd.to_pickle(trade_cis,              os.path.join(SAVE_DIR, 'combined_cpcv_trade_cis.pkl'))
pd.to_pickle(WEIGHTS,                os.path.join(SAVE_DIR, 'combined_cpcv_weights_config.pkl'))
pd.to_pickle(isolation_results,      os.path.join(SAVE_DIR, 'combined_cpcv_isolation.pkl'))
pd.to_pickle(sweep_rows,             os.path.join(SAVE_DIR, 'combined_cpcv_split_sweep.pkl'))
pd.to_pickle(inclusion_df,           os.path.join(SAVE_DIR, 'combined_cpcv_inclusion.pkl'))
pd.to_pickle(yearly,                 os.path.join(SAVE_DIR, 'combined_cpcv_yearly.pkl'))

print(f'All combined CPCV results saved → {SAVE_DIR}/')